In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
from torch.utils.tensorboard import SummaryWriter
from timm.models.vision_transformer import vit_base_patch16_224
from tqdm import tqdm

# ===================== Setup =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "/kaggle/input/indoor-scenes-cvpr-2019/indoorCVPR_09/Images"
# /kaggle/input/indoor-scenes-cvpr-2019/indoorCVPR_09
log_dir = "runs/vit_indoor"
os.makedirs(log_dir, exist_ok=True)
writer = SummaryWriter(log_dir=log_dir)

# ===================== Augmentations =====================
train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ===================== Dataset & Dataloader =====================
full_dataset = ImageFolder(data_dir)
class_names = full_dataset.classes
num_classes = len(class_names)

# Split into train and val
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)

# ===================== Model =====================
model = vit_base_patch16_224(pretrained=True)
model.head = nn.Linear(model.head.in_features, num_classes)
model = model.to(device)

# ===================== Optimizer, Loss, Scheduler =====================
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# ===================== Training Loop =====================
def train(epoch):
    model.train()
    total_loss, correct = 0, 0
    for i, (images, labels) in enumerate(tqdm(train_loader)):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

        if i == 0 and epoch == 0:
            img_grid = make_grid(images[:8].cpu())
            writer.add_image("Training Images", img_grid, 0)

    acc = 100 * correct / len(train_loader.dataset)
    writer.add_scalar("Loss/train", total_loss / len(train_loader), epoch)
    writer.add_scalar("Accuracy/train", acc, epoch)
    print(f"Epoch {epoch}: Train Loss: {total_loss / len(train_loader):.4f}, Train Acc: {acc:.2f}%")

# ===================== Validation Loop =====================
def validate(epoch):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

    acc = 100 * correct / len(val_loader.dataset)
    writer.add_scalar("Loss/val", total_loss / len(val_loader), epoch)
    writer.add_scalar("Accuracy/val", acc, epoch)
    print(f"Epoch {epoch}: Val Loss: {total_loss / len(val_loader):.4f}, Val Acc: {acc:.2f}%")

# ===================== Train =====================
num_epochs = 25
for epoch in range(num_epochs):
    train(epoch)
    validate(epoch)
    scheduler.step()

# ===================== Save Model =====================
torch.save(model.state_dict(), "vit_indoor_final.pth")
writer.close()


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

100%|██████████| 196/196 [3:24:48<00:00, 62.69s/it]  

Epoch 0: Train Loss: 3.5206, Train Acc: 12.69%


Epoch 0: Val Loss: 3.3890, Val Acc: 16.90%


 16%|█▌        | 31/196 [30:49<2:38:20, 57.58s/it]